## PII Redactor Example Notebook


**Author**: Pooja Holkar ,
**email**:poholkar@in.ibm.com

### What is a PII Redactor?

A PII (Personally Identifiable Information) Redactor is a tool designed to identify and redact sensitive information in text data. PII includes details that can be used to identify an individual, such as:

Names,
Email addresses,
Phone numbers,
Addresses,
Financial details (e.g., credit card numbers).

### Overview of the use case
In this usecase, the PII Redactor is applied to text extracted from invoices to ensure sensitive customer information is not exposed during processing, sharing, or storage.

 **Workflow Overview**

- **Extracting and Converting Text:** The content of the invoice, originally in PDF format, is processed using the docling2parquet transform to extract the text and convert it into a structured Parquet file, enabling easier handling and downstream processing.

- **Redacting Sensitive Information:** The generated Parquet file serves as the input for the dpk_pii_redactor transform. This step scans the invoice data for personally identifiable information (PII) and applies masking techniques to redact any sensitive content, ensuring data privacy and compliance.

- **Creating the Final Output:** After the redaction process, a new output Parquet file is generated in **output-redacted** folder, containing the same structured data as the original but with all sensitive details securely masked to prevent unauthorized access or exposure.


### Why is PII Redaction Important?

 **Data Privacy Compliance**: Adheres to regulations like GDPR, HIPAA, or CCPA that mandate safeguarding customer information.

 **Risk Mitigation**: Prevents unauthorized access to or misuse of sensitive data.

 **Automation Benefits**: Simplifies and accelerates the process of securing information in large-scale document handling.


## How to run this notebook

If you have python 3.11 or higher on your machine, you can also download the notebook and run it locally using a local python environment setup as follows:

```
python -m venv venv
source venv/bin/activate
pip install jupyterlab
jupyter lab Run_your_first_PII_redactor_transform.ipynb
```

For more advanced setup, please see setup [guide](https://github.com/data-prep-kit/data-prep-kit/blob/dev/doc/quick-start/quick-start.md).

An earlier version of this notebook was tested successfully on Google Colab. However, continuous changes in the Colab environment could introduce unexpected behavior/breakage. If you wish to try the Colab environment, click on [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data-prep-kit/data-prep-kit/blob/dev/recipes/PII/Run_your_first_PII_redactor_transform.ipynb).


### Pre-req: Install data-prep-kit toolkit

In [ ]:
%%capture cap --no-stderr
%pip install 'data-prep-toolkit-transforms[language]'
import pyarrow.parquet as pq
import pandas as pd
import os

## Step 1: Configuration

### Download Data and set input and output directories
#### We will place the downloaded files in the `input_dir`. For our use case, we have used Invoice data file that will undergo processing. The output for each transform run will be generated in separate directories, with directory names following the format `files_<transform_name>`, making it easy to verify the respective transform outputs. This concludes the setup section.

In [ ]:
import urllib.request
import shutil
shutil.os.makedirs("tmp/input", exist_ok=True)
urllib.request.urlretrieve("https://raw.githubusercontent.com/data-prep-kit/data-prep-kit/dev/examples/notebooks/PII/input-data/Invoice.pdf", "tmp/input/Invoice.pdf")

input_dir = "tmp/input"
output_dir = "output"
output_docling2pq_dir = os.path.join (output_dir, 'files_docling2parquet')
output_piiredactor_dir = os.path.join (output_dir, 'files_piiredacted')

## Step 2: Invoke Docling2Parquet transform to proces pdf files

In [ ]:
%%time

from dpk_docling2parquet import Docling2Parquet
from dpk_docling2parquet import docling2parquet_contents_types

STAGE = 1
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{input_dir}' --> output='{output_docling2pq_dir}'\n", flush=True)

Docling2Parquet(input_folder= input_dir,
               output_folder= output_docling2pq_dir,
               data_files_to_use=['.pdf'],
               docling2parquet_contents_type='text/markdown').transform()

## Step 3: Invoke PII Redactor configuration transform

In [ ]:
%%time

from dpk_pii_redactor import PIIRedactor

STAGE = 2
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_docling2pq_dir}' --> output='{output_piiredactor_dir}'\n", flush=True)
PIIRedactor(input_folder=output_docling2pq_dir,
            output_folder= output_piiredactor_dir,
            pii_redactor_entities = ["PERSON", "EMAIL_ADDRESS","ORGANIZATION","PHONE_NUMBER", "LOCATION"],
            pii_redactor_operator = "replace",
            pii_redactor_transformed_contents = "title").transform()

## Step 4: Display Output in a Readable Format with masked PII information

In [ ]:
data = pd.read_parquet('output/files_piiredacted/Invoice.parquet')
print(data["title"][0])
print(data["detected_pii"][0])



### This notebook effectively demonstrates how to seamlessly apply redaction for PII entities.